# Week 11 · Notebook 1: Extraction for the sex-prediction packages (MCD-rPPG)

**What this notebook does**
1. Writes the shared code (`sexpred_lib.py`) to Drive. Both package notebooks and the deployed predictor use these same functions.
2. Makes sure every front-camera session has a face colour trace (`roi_traces`). Missing ones are extracted from `raw/video`, or downloaded from Hugging Face.
3. Builds every input the packages need, for the full recording and for the first 120, 60 and 30 s:
   - rPPG features (heart rate, HRV, signal quality, skin colour)
   - pulse-shape features and region SNRs
   - appearance features
   - the 12-channel rPPG signal (4 face regions × POS, CHROM, green)
   - contact-PPG heart rate over the same seconds (used only to train the quality gate)
4. Checks that the code reproduces the training data and the video step.

**Which extraction was used for training.** The `roi_traces` files were written on 2026-10-11 between 09:44 and 18:13 UTC. Only `Week3_Stress_rPPG_600_FrontCam (3).ipynb` (created 09:33 that day) was open then, so its extractor is copied here unchanged. About 100 patients' traces date from 2026-10-03 and came from the Week 2 version of the same extractor, which that run skipped because the files existed. Section 5 re-extracts a few videos to show how closely today's extractor reproduces both groups.

**Outputs** (in `MCD_rPPG/sex_deploy/inputs/`): `features_<length>.csv` and `signals_<length>.npz` for `full`, `120s`, `60s`, `30s`, plus `verification_*.csv`.

**Run time (Colab CPU is enough):** section 3 builds inputs in about 15–40 min the first time (it saves progress every 200 sessions and resumes). Section 5 adds about 2–4 min per re-extracted video.

In [ ]:
# ---- CONFIG: edit this cell only ----
import os
ROOT = os.environ.get('MCD_ROOT', '/content/drive/MyDrive/MCD_rPPG')
DEPLOY = os.environ.get('SEX_DEPLOY', f'{ROOT}/sex_deploy')
LENGTHS = [None, 120, 60, 30]     # seconds of video used; None = whole recording (about 3 min)
EXTRACT_MISSING = True            # extract roi_traces for sessions without one (raw/video first, then Hugging Face)
N_VERIFY_VIDEOS = 4               # sessions re-extracted from video in section 5 (0 to skip)
WORKERS = None                    # processes for section 3; None = all CPU cores
QUICK = False                     # True = 60 people and 2 lengths, to check the notebook runs
if QUICK: LENGTHS, N_VERIFY_VIDEOS = [None, 30], 1
if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# OpenCV 4.x with the Haar face detector, as in Week 3.
import subprocess, sys
try:
    import cv2
    ok = hasattr(cv2, 'CascadeClassifier')
except ImportError:
    ok = False
if not ok:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless<5'])
    print('>>> Installed OpenCV 4.x. Runtime > Restart session, then run from the top. <<<')
else:
    print('OpenCV', cv2.__version__)

>>> Installed OpenCV 4.x. Runtime > Restart session, then run from the top. <<<


In [ ]:
# ---- shared code (sexpred_lib.py): written to Drive so every notebook and the deployed package use the same functions ----
LIB_SRC = r'''"""sexpred_lib: MCD-rPPG sex prediction, from face video to a calibrated probability.

Sources of each part, kept identical so deployed predictions match the training data:
- Video -> ROI colour traces: Week3_Stress_rPPG_600_FrontCam (3).ipynb (the notebook that wrote roi_traces on 2026-10-11).
- Session rPPG features (HRV, quality, skin colour): the same Week 3 notebook.
- 12-channel signals (POS, CHROM, green x 4 regions), appearance, pulse shape, random convolutions, CNN: Week 9.
- Quality gate: Week 7.
"""
import json, re
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks, resample_poly, welch
from scipy.stats import kurtosis, skew

LIB_VERSION = '1.0'
FS = 30.0
HR_BAND = (0.7, 3.5)
ROI_NAMES = ['face', 'forehead', 'left_cheek', 'right_cheek']
ROI_BOXES = {'face': (0.15, 0.85, 0.10, 0.90), 'forehead': (0.25, 0.75, 0.05, 0.25),
             'left_cheek': (0.12, 0.38, 0.50, 0.75), 'right_cheek': (0.62, 0.88, 0.50, 0.75)}
ALGOS = ['pos', 'chrom', 'green']
CHANNELS = [f'{r}_{a}' for r in ROI_NAMES for a in ALGOS]
POS_IDX = [CHANNELS.index(f'{r}_pos') for r in ROI_NAMES]
PHYS = ['rppg_hr', 'rppg_hr_beats', 'rppg_sdnn', 'rppg_rmssd', 'rppg_pnn50', 'rppg_ln_sdnn', 'rppg_ln_rmssd', 'rppg_ln_lf', 'rppg_ln_hf',
        'rppg_lf_hf', 'rppg_rise_frac', 'rppg_half_width']
QUAL = ['rppg_snr', 'rppg_snr_face', 'rppg_snr_forehead', 'rppg_snr_left_cheek', 'rppg_snr_right_cheek', 'rppg_hr_win_sd',
        'rppg_hr_consistency', 'rppg_n_beats', 'rppg_ibi_ok_frac', 'rppg_beat_consistency']
SKIN = ['rppg_skin_r', 'rppg_skin_g', 'rppg_skin_lum']
MORPH_COLS = ['rise_frac', 'w25', 'w50', 'w75', 'max_slope_pos', 'slope_ratio', 'area_ratio', 'skew', 'kurt', 'apg_b_a', 'apg_late_a',
              'h2_h1', 'h3_h1', 'h4_h1', 'consistency', 'kept_frac', 'beats_per_min']
SHAPE_COLS = [f'shape_{c}' for c in MORPH_COLS] + [f'snr8_{r}' for r in ROI_NAMES]
TAB_COLS = PHYS + QUAL + SHAPE_COLS                       # rPPG tabular model
APP_COLS = [f'{r}_{s}' for r in ROI_NAMES for s in ('chroma_r', 'chroma_g', 'lum', 'lum_var')] + SKIN   # appearance model
GATE_COLS = ['rppg_face_missing_frac', 'rppg_fps_measured', 'rppg_snr_face', 'rppg_snr_forehead', 'rppg_snr_left_cheek',
             'rppg_snr_right_cheek', 'rppg_snr', 'rppg_hr_win_sd', 'rppg_hr_consistency', 'rppg_n_beats', 'rppg_ibi_ok_frac',
             'rppg_beat_consistency', 'rppg_skin_lum', 'hr_beat_gap']
_trapz = getattr(np, 'trapezoid', None) or np.trapz
WIN = int(20 * FS)   # CNN and random-conv window: 20 s

# =====================================================================================
# 1. Video -> ROI colour traces (Week 3 extractor, unchanged)
# =====================================================================================
def cascade_path(tmp_dir='/tmp'):
    import cv2
    name = 'haarcascade_frontalface_default.xml'
    root = getattr(getattr(cv2, 'data', None), 'haarcascades', None)
    if root and (Path(root) / name).exists():
        return str(Path(root) / name)
    dst = Path(tmp_dir) / name
    if not dst.exists():
        import urllib.request
        urllib.request.urlretrieve(f'https://raw.githubusercontent.com/opencv/opencv/4.x/data/haarcascades/{name}', dst)
    return str(dst)

def extract_roi_traces(video_path, out_path=None, detect_every=10, ema=0.3, scale=0.5, cascade=None):
    """Per-frame skin-masked mean RGB for face, forehead, left cheek and right cheek. Returns the same keys as roi_traces/*.npz."""
    import cv2
    cv2.setNumThreads(1)
    det = cv2.CascadeClassifier(cascade or cascade_path())
    if det.empty():
        raise RuntimeError('Haar face detector failed to load (OpenCV build without it?)')
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'cannot open video {video_path}')
    fps_c = cap.get(cv2.CAP_PROP_FPS)
    traces, dup, found, box, prev_small = [], [], [], None, None
    i = 0
    while True:
        ok, bgr = cap.read()
        if not ok:
            break
        small = cv2.resize(bgr, (64, 48), interpolation=cv2.INTER_AREA)
        dup.append(prev_small is not None and np.array_equal(small, prev_small)); prev_small = small
        if i % detect_every == 0:
            g = cv2.resize(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), None, fx=scale, fy=scale)
            faces = det.detectMultiScale(g, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
            d = None
            if len(faces):
                x, y, w, h = max(faces, key=lambda b: b[2] * b[3])
                d = np.array([x, y, w, h], float) / scale
                box = d if box is None else (1 - ema) * box + ema * d
            found.append(d is not None)
        else:
            found.append(found[-1] if found else False)
        row = np.full((len(ROI_NAMES), 3), np.nan, np.float32)
        if box is not None:
            x, y, w, h = box
            rgb = bgr[..., ::-1]
            for k, n in enumerate(ROI_NAMES):
                fx0, fx1, fy0, fy1 = ROI_BOXES[n]
                xa, xb = int(max(0, x + fx0 * w)), int(min(bgr.shape[1], x + fx1 * w))
                ya, yb = int(max(0, y + fy0 * h)), int(min(bgr.shape[0], y + fy1 * h))
                if xb - xa < 4 or yb - ya < 4:
                    continue
                patch = rgb[ya:yb, xa:xb]
                ycc = cv2.cvtColor(bgr[ya:yb, xa:xb], cv2.COLOR_BGR2YCrCb)
                skin = (ycc[..., 1] >= 133) & (ycc[..., 1] <= 173) & (ycc[..., 2] >= 77) & (ycc[..., 2] <= 127)
                px = patch[skin] if skin.sum() > 0.2 * skin.size else patch.reshape(-1, 3)
                row[k] = px.mean(0)
        traces.append(row); i += 1
    cap.release()
    tr = dict(traces=np.array(traces, np.float32), roi_names=np.array(ROI_NAMES), dup=np.array(dup), face_found=np.array(found),
              fps_container=fps_c)
    if out_path is not None:
        np.savez_compressed(out_path, **tr)
    return tr

def load_traces(path):
    with np.load(path) as z:
        return {k: z[k] for k in z.files}

TS_RE = re.compile(r'(\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}(?:[.,]\d+)?)')
def parse_meta_timestamps(path):
    """Per-frame timestamps (seconds from start) from an MCD meta file, or None."""
    if path is None or not Path(path).exists():
        return None
    lines = Path(path).read_text(errors='ignore').splitlines()
    ts = [pd.Timestamp(m.group(1).replace(',', '.')).value / 1e9 for m in map(TS_RE.search, lines) if m]
    if len(ts) < 100:
        ts = []
        for ln in lines:
            nums = re.findall(r'[-+]?\d+(?:\.\d+)?', ln)
            if len(nums) == 1:
                ts.append(float(nums[0]))
    if len(ts) < 100:
        return None
    t = np.asarray(ts, float); t = t - t[0]; d = np.diff(t)
    if np.mean(d < 0) > 0.01:
        return None
    if np.median(d) > 1:
        t = t / 1000.0
    return t

def time_base(N, ts, fps_hdr):
    """Week 3 rule: meta timestamps if they match the frame count, else the meta fps, else the container fps."""
    ts = np.array([]) if ts is None else np.asarray(ts, float)
    if len(ts) >= 100 and abs(len(ts) - N) <= 5:
        t = ts[:N]
        if len(t) < N:
            t = np.concatenate([t, t[-1] + np.arange(1, N - len(t) + 1) * np.median(np.diff(t))])
        return t, 1
    if len(ts) >= 100:
        return np.arange(N) / ((len(ts) - 1) / (ts[-1] - ts[0])), 2
    fps = fps_hdr if 5 < fps_hdr < 121 else 30.0
    return np.arange(N) / fps, 0

def _prepared(tr, ts, max_seconds=None):
    rgb = tr['traces'].astype(float); dup = tr['dup'].astype(bool); N = len(rgb)
    t, used = time_base(N, ts, float(tr['fps_container']))
    if max_seconds is not None:
        n = int(np.searchsorted(t, t[0] + max_seconds, side='right'))
        rgb, dup, t = rgb[:n], dup[:n], t[:n]
    return rgb, dup, t, used

# =====================================================================================
# 2. Week 3 session features (rPPG part)
# =====================================================================================
def bandpass(x, fs, lo, hi, order=3):
    b, a = butter(order, [lo / (fs / 2), hi / (fs / 2)], btype='band'); return filtfilt(b, a, x)

def bandpass_nd(x, fs, lo, hi, order=3):
    b, a = butter(order, [lo / (fs / 2), min(hi / (fs / 2), .95)], btype='band'); return filtfilt(b, a, x, axis=-1)

def interp_nans(a):
    a = np.array(a, float, copy=True); flat = a.reshape(len(a), -1); idx = np.arange(len(a))
    for j in range(flat.shape[1]):
        m = np.isfinite(flat[:, j])
        if m.sum() < 0.5 * len(m):
            return None
        flat[~m, j] = np.interp(idx[~m], idx[m], flat[m, j])
    return flat.reshape(a.shape)

def _windows(rgb, fs, win):
    from numpy.lib.stride_tricks import sliding_window_view
    l = int(round(win * fs)); W = sliding_window_view(rgb, l, axis=0); mu = W.mean(axis=2, keepdims=True); mu[mu == 0] = 1e-9
    return W / mu, l

def _overlap_add(h, N, l):
    h = h - h.mean(axis=1, keepdims=True); H = np.zeros(N)
    for j in range(l):
        H[j:j + len(h)] += h[:, j]
    return H

def pos(rgb, fs, win=1.6):
    """POS (Wang et al. 2017). Vectorised; identical output to the Week 3 loop version."""
    C, l = _windows(rgb, fs, win); S0 = C[:, 1] - C[:, 2]; S1 = -2 * C[:, 0] + C[:, 1] + C[:, 2]; sd1 = S1.std(axis=1, keepdims=True)
    return _overlap_add(S0 + (S0.std(axis=1, keepdims=True) / np.where(sd1 > 0, sd1, np.inf)) * S1, len(rgb), l)

def chrom(rgb, fs, win=1.6):
    C, l = _windows(rgb, fs, win); X = 3 * C[:, 0] - 2 * C[:, 1]; Y = 1.5 * C[:, 0] + C[:, 1] - 1.5 * C[:, 2]; sy = Y.std(axis=1, keepdims=True)
    return _overlap_add(X - (X.std(axis=1, keepdims=True) / np.where(sy > 0, sy, np.inf)) * Y, len(rgb), l)

def green_norm(rgb, fs, win=1.6):
    l = int(round(win * fs)); g = rgb[:, 1]; m = np.convolve(g, np.ones(l) / l, mode='same'); return g / np.where(m == 0, 1e-9, m) - 1

def spectrum(x, fs, seg_s=12):
    nper = int(min(len(x), seg_s * fs)); return welch(x, fs, nperseg=nper, noverlap=nper // 2, nfft=max(8192, nper))

def hr_harmonic(x, fs, band=HR_BAND):
    f, P = spectrum(x, fs); m = (f >= band[0]) & (f <= band[1])
    score = P[m] + 0.5 * np.interp(2 * f[m], f, P, right=0); return 60 * f[m][np.argmax(score)], f, P

def snr_db(f, P, f0, half=0.1):
    sig = (np.abs(f - f0) <= half) | (np.abs(f - 2 * f0) <= half); rng = (f >= 0.6) & (f <= 4.0)
    num, den = P[sig & rng].sum(), P[~sig & rng].sum()
    return 10 * np.log10(num / den) if num > 0 and den > 0 else np.nan

def detect_beats(x, fs, hr_bpm):
    period = fs * 60.0 / hr_bpm; pk, _ = find_peaks(x, distance=max(1, int(0.6 * period)), prominence=0.1 * np.std(x)); return pk

def hrv_from_peaks(pk, fs, min_dur_freq=100):
    t = pk / fs; ibi = np.diff(t)
    if len(ibi) < 10:
        return {}
    tb = t[1:]; med = pd.Series(ibi).rolling(7, center=True, min_periods=1).median().values
    ok = (ibi > 0.33) & (ibi < 1.5) & (np.abs(ibi - med) < 0.2 * med)
    if ok.sum() < 10:
        return dict(ibi_ok_frac=ok.mean())
    ic, tc = ibi[ok], tb[ok]; d = np.diff(ibi)[ok[1:] & ok[:-1]]
    o = dict(n_beats=len(pk), ibi_ok_frac=ok.mean(), hr_beats=60 / ic.mean(), sdnn=1000 * ic.std(ddof=1),
             rmssd=1000 * np.sqrt(np.mean(d ** 2)) if len(d) else np.nan, pnn50=np.mean(np.abs(d) > 0.05) if len(d) else np.nan)
    o['ln_sdnn'] = np.log(max(o['sdnn'], 1e-3)); o['ln_rmssd'] = np.log(max(o['rmssd'], 1e-3)) if np.isfinite(o['rmssd']) else np.nan
    if tc[-1] - tc[0] >= min_dur_freq and len(ic) >= 60:
        g = np.arange(tc[0], tc[-1], 0.25); y = np.interp(g, tc, ic); y = y - y.mean(); f, P = welch(y, 4.0, nperseg=min(256, len(y)))
        lf = _trapz(P[(f >= 0.04) & (f < 0.15)], f[(f >= 0.04) & (f < 0.15)]) * 1e6; hf = _trapz(P[(f >= 0.15) & (f < 0.40)], f[(f >= 0.15) & (f < 0.40)]) * 1e6
        o.update(lf=lf, hf=hf, ln_lf=np.log(max(lf, 1e-6)), ln_hf=np.log(max(hf, 1e-6)), lf_hf=lf / hf if hf > 0 else np.nan)
    return o

def morphology(x, pk, fs):
    if len(pk) < 10:
        return {}
    T = int(np.median(np.diff(pk))); a, b = int(0.6 * T), int(0.4 * T)
    segs = [x[p - a:p + b] for p in pk if p - a >= 0 and p + b <= len(x)]
    if len(segs) < 10:
        return {}
    S = np.array(segs); S = (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-12); tpl = S.mean(0)
    cons = np.mean([np.corrcoef(s, tpl)[0, 1] for s in S]); i_min = int(np.argmin(tpl[:a + 1])); half = (tpl.max() + tpl.min()) / 2
    return dict(beat_consistency=cons, rise_frac=(a - i_min) / T, half_width=np.mean(tpl > half))

def windowed_hr(x, fs, win=10, step=2):
    n, s = int(win * fs), int(step * fs); return np.array([hr_harmonic(x[i:i + n], fs)[0] for i in range(0, len(x) - n + 1, s)])

def pulse_features(x, fs):
    hr, f, P = hr_harmonic(x, fs); o = dict(hr=hr, snr=snr_db(f, P, hr / 60)); w = windowed_hr(x, fs)
    if len(w):
        o['hr_win_sd'] = np.std(w); o['hr_consistency'] = np.mean(np.abs(w - hr) < 5)
    pk = detect_beats(x, fs, hr); o.update(hrv_from_peaks(pk, fs)); o.update(morphology(x, pk, fs)); return o

def rppg_features(tr, ts=None, max_seconds=None):
    """Week 3 rppg_features (POS per ROI, 0.7-3.5 Hz, SNR-weighted combination). Keys carry the rppg_ prefix."""
    rgb, dup, t, used = _prepared(tr, ts, max_seconds); N = len(rgb)
    names = [str(n) for n in tr['roi_names']] if 'roi_names' in tr else ROI_NAMES
    o = dict(face_missing_frac=float(np.mean(~np.isfinite(rgb[:, 0, 0]))), dup_frac=float(dup.mean()))
    rgb = rgb.copy(); rgb[dup] = np.nan
    o['used_ts'] = used; o['fps_measured'] = (N - 1) / (t[-1] - t[0]); o['duration_s'] = t[-1] - t[0]
    tu = np.arange(t[0], t[-1], 1 / FS); sigs, snrs = [], []
    for k, name in enumerate(names):
        c = interp_nans(rgb[:, k, :])
        if c is None:
            continue
        cu = np.column_stack([np.interp(tu, t, c[:, j]) for j in range(3)])
        s = bandpass(pos(cu, FS), FS, *HR_BAND); hr_k, f, P = hr_harmonic(s, FS); snr_k = snr_db(f, P, hr_k / 60)
        o[f'snr_{name}'] = snr_k; sigs.append(s / (np.std(s) + 1e-12)); snrs.append(snr_k if np.isfinite(snr_k) else -20)
        if name == 'face':
            m = c.mean(0); o.update(skin_r=m[0] / m.sum(), skin_g=m[1] / m.sum(), skin_lum=m.mean())
    if sigs:
        w = 10 ** (np.array(snrs) / 10); comb = np.sum([wi * si for wi, si in zip(w, sigs)], axis=0) / w.sum()
        o.update(pulse_features(comb, FS))
    o = {f'rppg_{k}': v for k, v in o.items()}
    if 'rppg_hr' in o and 'rppg_hr_beats' in o:
        o['hr_beat_gap'] = abs(o['rppg_hr'] - o['rppg_hr_beats'])
    return o

def load_pw(path):
    vals = []
    with open(path, errors='ignore') as fh:
        for ln in fh:
            tok = ln.split()
            if tok:
                try: vals.append(float(tok[0]))
                except ValueError: pass
    return np.asarray(vals)

def contact_hr(pw_path, fs=100.0, max_seconds=None):
    """Contact-PPG heart rate (Week 3), optionally over the first max_seconds. Used only as the label for training the quality gate."""
    from scipy.signal import detrend
    x = load_pw(pw_path)
    if max_seconds is not None:
        x = x[:int(max_seconds * fs)]
    if len(x) < min(30, max_seconds or 30) * fs - 1:
        return np.nan
    return hr_harmonic(bandpass(detrend(x), fs, *HR_BAND), fs)[0]

# =====================================================================================
# 3. Week 9 signals: 4 regions x (POS, CHROM, green), appearance, pulse shape
# =====================================================================================
def multi_signals(tr, ts=None, max_seconds=None):
    """12 rPPG channels at 30 Hz, 0.3-12 Hz, z-scored (float16), and 16 appearance values. None if a region is mostly missing."""
    rgb, dup, t, _ = _prepared(tr, ts, max_seconds); rgb = rgb.copy(); rgb[dup] = np.nan
    tu = np.arange(t[0], t[-1], 1 / FS); sig, app = [], []
    for k in range(rgb.shape[1]):
        c = interp_nans(rgb[:, k, :])
        if c is None:
            return None, None
        cu = np.column_stack([np.interp(tu, t, c[:, j]) for j in range(3)])
        for fn in (pos, chrom, green_norm):
            s = bandpass_nd(fn(cu, FS), FS, .3, 12.); sig.append(s / (s.std() + 1e-12))
        tot = cu.sum(1) + 1e-9; lum = cu.mean(1); sec = lum[:len(lum) // 30 * 30].reshape(-1, 30).mean(1)
        app += [(cu[:, 0] / tot).mean(), (cu[:, 1] / tot).mean(), lum.mean(), sec.std() / (sec.mean() + 1e-9)]
    return np.stack(sig).astype(np.float16), np.array(app, np.float32)

def views(multi_raw):
    """The two model inputs from the stored 12 channels: all 12 at 0.5-8 Hz (CNN) and the 4 POS channels at 0.5-8 Hz (random conv, pulse shape)."""
    a = multi_raw.astype(np.float32)
    return bandpass_nd(a, FS, .5, 8.).astype(np.float32), bandpass_nd(a[POS_IDX], FS, .5, 8.).astype(np.float32)

def hr_est(x, fs=FS):
    y = bandpass_nd(x, fs, .7, 3.5); f, Pw = welch(y, fs, nperseg=min(len(y), int(12 * fs)), nfft=8192)
    m = (f >= .7) & (f <= 3.5); sc = Pw[m] + .5 * np.interp(2 * f[m], f, Pw, right=0); return f[m][np.argmax(sc)]

def snr_db8(x, f0, fs=FS):
    f, Pw = welch(bandpass_nd(x, fs, .5, 4.5), fs, nperseg=min(len(x), int(12 * fs)), nfft=8192)
    sig = ((np.abs(f - f0) <= .1) | (np.abs(f - 2 * f0) <= .1)) & (f >= .6) & (f <= 4.); rng = (f >= .6) & (f <= 4.)
    return 10 * np.log10(Pw[sig].sum() / max(Pw[rng & ~sig].sum(), 1e-12))

def combine(a):
    snr = np.array([snr_db8(ch, hr_est(ch)) for ch in a]); w = 10 ** (np.clip(np.nan_to_num(snr, nan=-20), -20, 20) / 10)
    x = (w[:, None] * a).sum(0) / w.sum(); return x / (x.std() + 1e-12), snr

def pulse_template(x, fs=FS, up=4, n=64):
    f0 = hr_est(x, fs); xu = resample_poly(x, up, 1); xn = resample_poly(bandpass_nd(x, fs, .7, 3.5), up, 1); T = fs * up / f0
    pk, _ = find_peaks(xn, distance=max(1, int(.6 * T))); a, b = int(.45 * T), int(.55 * T)
    segs = np.array([xu[p - a:p + b] for p in pk if p - a >= 0 and p + b <= len(xu)])
    if len(segs) < 10:
        return None
    segs = (segs - segs.mean(1, keepdims=True)) / (segs.std(1, keepdims=True) + 1e-9); med = np.median(segs, 0)
    r = np.array([np.corrcoef(s, med)[0, 1] for s in segs]); keep = r > .5
    if keep.sum() < 10:
        keep = np.ones(len(segs), bool)
    tpl = segs[keep].mean(0); tpl = np.roll(tpl, -int(np.argmin(tpl[:int(.7 * len(tpl))])))
    t = np.interp(np.linspace(0, len(tpl) - 1, n), np.arange(len(tpl)), tpl); t = (t - t.mean()) / (t.std() + 1e-9)
    d1 = np.gradient(t); d2 = np.gradient(d1); pkp = int(np.argmax(t)); rng_ = t.max() - t.min(); base = t - t.min()
    ia = int(np.argmax(d2[:max(pkp, 2)])); ib = ia + int(np.argmin(d2[ia:max(pkp + 4, ia + 2)])); late = d2[pkp:int(.75 * n)]
    f, Pw = welch(x, fs, nperseg=min(len(x), int(12 * fs)), nfft=8192); h = [Pw[np.abs(f - k * f0) <= .08].sum() + 1e-12 for k in (1, 2, 3, 4)]
    return dict(rise_frac=pkp / n, w25=np.mean(t > t.min() + .75 * rng_), w50=np.mean(t > t.min() + .5 * rng_), w75=np.mean(t > t.min() + .25 * rng_),
                max_slope_pos=np.argmax(d1) / n, slope_ratio=d1.max() / max(-d1.min(), 1e-9), area_ratio=base[:pkp + 1].sum() / max(base[pkp:].sum(), 1e-9),
                skew=skew(t), kurt=kurtosis(t), apg_b_a=d2[ib] / max(abs(d2[ia]), 1e-9), apg_late_a=(late.max() if len(late) else np.nan) / max(abs(d2[ia]), 1e-9),
                h2_h1=np.log10(h[1] / h[0]), h3_h1=np.log10(h[2] / h[0]), h4_h1=np.log10(h[3] / h[0]), consistency=r[keep].mean(), kept_frac=keep.mean(),
                beats_per_min=60 * f0)

def shape_features(pos4):
    """Pulse-shape features of the SNR-weighted average of the 4 POS channels, plus each region's SNR."""
    out = {c: np.nan for c in SHAPE_COLS}
    try:
        x, snr = combine(pos4)
        for r, s in zip(ROI_NAMES, snr): out[f'snr8_{r}'] = s
        f = pulse_template(x)
        if f is not None:
            out.update({f'shape_{k}': v for k, v in f.items()})
    except Exception:
        pass
    return out

# =====================================================================================
# 4. Everything one session needs, from traces
# =====================================================================================
def session_record(tr, ts=None, max_seconds=None):
    """All features of one session (rPPG, pulse shape, appearance) and the stored 12-channel signal (float16), or None if unusable."""
    feats = rppg_features(tr, ts, max_seconds)
    raw, app = multi_signals(tr, ts, max_seconds)
    if raw is None:
        return feats, None
    feats.update(shape_features(views(raw)[1])); feats.update(dict(zip(APP_COLS[:16], app)))
    return feats, raw

def session_inputs(tr, ts=None, max_seconds=None):
    """Returns (features dict, 12-channel CNN input, 4 POS channels for random conv). The arrays are None if the session is unusable."""
    feats, raw = session_record(tr, ts, max_seconds)
    if raw is None:
        return feats, None, None
    m12, p4 = views(raw)
    return feats, m12, p4

def usable(feats, raw):
    return raw is not None and np.isfinite(feats.get('rppg_hr', np.nan)) and raw.shape[-1] >= WIN

# =====================================================================================
# 5. Random convolutions (Week 9)
# =====================================================================================
_KERN = {}
def kernels(L, n_kernels, seed=0):
    key = (L, n_kernels, seed)
    if key not in _KERN:
        rg = np.random.default_rng(seed); nfft = 2 * L; ks, dil = [], []
        for _ in range(n_kernels):
            k = rg.normal(size=rg.choice([7, 9, 11])); k -= k.mean(); d = int(2 ** rg.uniform(0, np.log2((L - 1) / (len(k) - 1)) * .55))
            kk = np.zeros((len(k) - 1) * d + 1); kk[::d] = k; ks.append(np.fft.rfft(kk, nfft)); dil.append(len(kk))
        _KERN[key] = (ks, dil)
    return _KERN[key]

def rand_features(sigs, n_kernels=500, win_s=20, n_win=6):
    """sigs: list of 1-D signals (or None). Returns (len(sigs), 2*n_kernels) array, NaN rows for unusable signals."""
    L = int(FS * win_s); nfft = 2 * L; ks, dil = kernels(L, n_kernels); windows, owner = [], []
    for i, x in enumerate(sigs):
        if x is None or len(x) < L:
            continue
        x = np.nan_to_num(np.asarray(x, np.float32))
        for s in np.linspace(0, len(x) - L, n_win).astype(int):
            w = x[s:s + L]; windows.append((w - w.mean()) / (w.std() + 1e-6)); owner.append(i)
    out = np.full((len(sigs), 2 * n_kernels), np.nan, np.float32)
    if not windows:
        return out
    F = np.fft.rfft(np.stack(windows), nfft, axis=1); feat = np.zeros((len(windows), 2 * n_kernels), np.float32)
    for j, (K, l) in enumerate(zip(ks, dil)):
        c = np.fft.irfft(F * K, nfft, axis=1)[:, l - 1:L]; feat[:, 2 * j] = (c > 0).mean(1); feat[:, 2 * j + 1] = c.max(1)
    g = pd.DataFrame(feat).groupby(np.array(owner)).mean()
    out[g.index.values] = g.values
    return out

def rc_session_features(pos4_list, n_kernels=500):
    """Random-conv features for many sessions: each region's POS channel separately, concatenated (4 x 2 x n_kernels)."""
    return np.hstack([rand_features([None if a is None else a[j] for a in pos4_list], n_kernels) for j in range(len(POS_IDX))])

# =====================================================================================
# 6. CNN (Week 9 ResNet1D, improved training)
# =====================================================================================
def _torch():
    import torch, torch.nn as nn
    return torch, nn

_NET = None
def make_net(in_ch):
    global _NET
    torch, nn = _torch()
    if _NET is None:
        class ResBlock(nn.Module):
            def __init__(self, i, o, stride=2, k=15):
                super().__init__()
                self.net = nn.Sequential(nn.Conv1d(i, o, k, stride=stride, padding=k // 2), nn.BatchNorm1d(o), nn.ReLU(), nn.Dropout(.1),
                                         nn.Conv1d(o, o, k, padding=k // 2), nn.BatchNorm1d(o))
                self.skip = nn.Sequential(nn.Conv1d(i, o, 1, stride=stride), nn.BatchNorm1d(o)); self.act = nn.ReLU()
            def forward(self, x): return self.act(self.net(x) + self.skip(x))
        class Net(nn.Module):
            def __init__(self, in_ch):
                super().__init__()
                self.enc = nn.Sequential(nn.Conv1d(in_ch, 32, 15, stride=2, padding=7), nn.BatchNorm1d(32), nn.ReLU(), ResBlock(32, 32), ResBlock(32, 64), ResBlock(64, 64))
                self.head = nn.Sequential(nn.Dropout(.4), nn.Linear(64, 1))
            def forward(self, x): return self.head(self.enc(x).mean(-1)).squeeze(-1)
        _NET = Net
    return _NET(in_ch)

def zwin(x): return (x - x.mean(-1, keepdims=True)) / (x.std(-1, keepdims=True) + 1e-6)

def _sample(items, sig, flag, rng, ch_group=3):
    C = sig[items[0]].shape[0]; X = np.empty((len(items), C + 1, WIN), np.float32); n_groups = C // ch_group
    for n, i in enumerate(items):
        T = sig[i].shape[1]; L = min(T, int(round(WIN * rng.uniform(.92, 1.08)))); st = rng.integers(0, T - L + 1); seg = sig[i][:, st:st + L].astype(np.float32)
        if L != WIN:
            x = np.linspace(0, L - 1, WIN); i0 = np.floor(x).astype(int); w = (x - i0).astype(np.float32); i1 = np.minimum(i0 + 1, L - 1); seg = seg[:, i0] * (1 - w) + seg[:, i1] * w
        seg = zwin(seg); seg = seg + rng.normal(0, rng.uniform(0, .3), seg.shape).astype(np.float32)
        if n_groups > 1 and rng.random() < .3:
            g = rng.integers(n_groups); seg[g * ch_group:(g + 1) * ch_group] = 0
        X[n, :C] = seg; X[n, C] = flag[i]
    return X

def cnn_session_logit(model, x, flag, device, stride_s=5):
    """Mean window logit for one session (x: channels x samples). NaN if shorter than 20 s."""
    torch, _ = _torch()
    if x is None or x.shape[1] < WIN:
        return np.nan
    st = np.arange(0, x.shape[1] - WIN + 1, int(stride_s * FS)); C = x.shape[0]; X = np.empty((len(st), C + 1, WIN), np.float32)
    for m, s in enumerate(st): X[m, :C] = zwin(x[:, s:s + WIN]); X[m, C] = flag
    model.eval()
    with torch.no_grad():
        return float(model(torch.from_numpy(X).to(device)).mean().cpu())

def train_cnn(sig, flag, y, ids, train_idx, seed, device, epochs=40, patience=8, per_session=8, batch=128, ch_group=3):
    """Week 9 improved training. sig: list of (12, T) arrays or None; train_idx: session indices. Early stopping on 10% of training people."""
    torch, nn = _torch()
    rng = np.random.default_rng(seed); torch.manual_seed(seed); people = pd.Series(y[train_idx], index=ids[train_idx]).groupby(level=0).first()
    val_ids = set(np.concatenate([rng.choice(people.index[people == c], max(3, int(.1 * (people == c).sum())), replace=False) for c in (0, 1)]))
    usable = [i for i in train_idx if sig[i] is not None and sig[i].shape[1] >= WIN]
    fit = np.array([i for i in usable if ids[i] not in val_ids]); val = np.array([i for i in usable if ids[i] in val_ids])
    model = make_net(sig[fit[0]].shape[0] + 1).to(device); pos_ = float(y[fit].mean()); pw = torch.tensor((1 - pos_) / pos_, dtype=torch.float32, device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    steps = epochs * int(np.ceil(len(fit) * per_session / batch)); sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, max(steps, 1))
    best, state, bad = -1., None, 0
    from sklearn.metrics import roc_auc_score
    for ep in range(epochs):
        model.train(); order = rng.permutation(np.repeat(fit, per_session))
        for b in range(0, len(order), batch):
            ch = order[b:b + batch]
            if len(ch) < 2: continue
            X = torch.from_numpy(_sample(ch, sig, flag, rng, ch_group)).to(device); t = torch.from_numpy((y[ch] * .9 + .05).astype(np.float32)).to(device)
            loss = nn.functional.binary_cross_entropy_with_logits(model(X), t, pos_weight=pw); opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        pv = np.array([cnn_session_logit(model, sig[i], flag[i], device) for i in val])
        f = pd.DataFrame(dict(i=ids[val], y=y[val], p=pv)).dropna().groupby('i').mean(); auc = roc_auc_score(f.y.astype(int), f.p)
        if auc > best + 1e-4: best, bad, state = auc, 0, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state); model.eval(); return model

# =====================================================================================
# 7. Tabular models and the quality gate
# =====================================================================================
def make_hgb():
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import make_pipeline
    return make_pipeline(SimpleImputer(strategy='median'), HistGradientBoostingClassifier(max_iter=200, max_depth=3, learning_rate=.04, min_samples_leaf=15,
                         l2_regularization=1., class_weight='balanced', random_state=0))

def make_rc_model():
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(C=.001, class_weight='balanced', max_iter=3000))

def make_gate():
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import make_pipeline
    return make_pipeline(SimpleImputer(strategy='median'), HistGradientBoostingClassifier(max_depth=3, max_iter=150, learning_rate=.05, random_state=0))

def two_session_row(before, after, cols):
    """One person row: before, after and change (after minus before) for each column."""
    b = np.array([before.get(c, np.nan) if before else np.nan for c in cols], float); a = np.array([after.get(c, np.nan) if after else np.nan for c in cols], float)
    return np.concatenate([b, a, a - b])

def single_session_row(sess, is_after, cols):
    return np.concatenate([np.array([sess.get(c, np.nan) for c in cols], float), [float(is_after)]])

def logit(p):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6); return np.log(p / (1 - p))

def sigmoid(z): return 1 / (1 + np.exp(-np.asarray(z, float)))

# =====================================================================================
# 8. Deployed predictor
# =====================================================================================
class SexPredictor:
    """Loads a package folder written by the package notebook and predicts from videos or ROI traces.

    predict(before_video=..., after_video=..., before_meta=..., after_meta=...)  # either session may be omitted
    Returns a dict with probability_male, prediction ('M', 'F' or None if no session passed the quality gate), mode, components and per-session quality.
    """
    def __init__(self, package_dir, device=None):
        import joblib
        self.dir = Path(package_dir); self.cfg = json.loads((self.dir / 'config.json').read_text())
        self.comp = self.cfg['components']; self.m = {}
        for name in ('rc', 'tab_two', 'tab_single', 'app_two', 'app_single', 'gate', 'calibrators'):
            p = self.dir / f'{name}.joblib'
            if p.exists(): self.m[name] = joblib.load(p)
        self.cnn = []
        if 'cnn' in self.comp:
            torch, _ = _torch(); self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
            for p in sorted(self.dir.glob('cnn_seed*.pt')):
                net = make_net(len(CHANNELS) + 1); net.load_state_dict(torch.load(p, map_location=self.device)); net.to(self.device).eval(); self.cnn.append(net)

    def session_from_video(self, video, meta=None, step='before', max_seconds=None):
        return self.session_from_traces(extract_roi_traces(video), parse_meta_timestamps(meta), step, max_seconds)

    def session_from_traces(self, tr, ts=None, step='before', max_seconds=None):
        f, m12, p4 = session_inputs(tr, ts, max_seconds)
        return dict(step=step, feats=f, m12=m12, p4=p4)

    def score_sessions(self, sessions, use_gate=True, threshold_kind=None):
        cfg = self.cfg; out = dict(sessions=[], components={}, package=cfg['package'], lib_version=LIB_VERSION)
        kept = []
        for s in sessions:
            f = s['feats']; q = dict(step=s['step'], usable=bool(usable(f, s['m12'])), duration_s=f.get('rppg_duration_s'))
            if q['usable'] and 'gate' in self.m:
                x = np.array([[f.get(c, np.nan) for c in GATE_COLS]], float); q['p_bad'] = float(self.m['gate'].predict_proba(x)[0, 1])
                q['passed_gate'] = q['p_bad'] < cfg['gate_threshold']
            else:
                q['passed_gate'] = q['usable']
            if q['usable'] and (q['passed_gate'] or not use_gate): kept.append(s)
            out['sessions'].append(q)
        if not kept:
            out.update(probability_male=None, prediction=None, mode=None, reason='no session passed the quality check'); return out
        steps = {s['step']: s for s in kept}; two = 'before' in steps and 'after' in steps; mode = 'two' if two else 'single'
        lg = []
        if 'cnn' in self.comp:
            ps = [sigmoid(np.nanmean([cnn_session_logit(net, s['m12'], float(s['step'] == 'after'), self.device) for net in self.cnn])) for s in kept]
            out['components']['cnn'] = float(np.nanmean(ps))
        if 'rc' in self.comp:
            X = rc_session_features([s['p4'] for s in kept], cfg['n_kernels']); out['components']['rc'] = float(np.mean(self.m['rc'].predict_proba(X)[:, 1]))
        for comp, cols in (('tab', TAB_COLS), ('app', APP_COLS)):
            if comp not in self.comp: continue
            if two:
                x = two_session_row(steps['before']['feats'], steps['after']['feats'], cols)[None]; p = self.m[f'{comp}_two'].predict_proba(x)[0, 1]
            else:
                p = float(np.mean([self.m[f'{comp}_single'].predict_proba(single_session_row(s['feats'], s['step'] == 'after', cols)[None])[0, 1] for s in kept]))
            out['components'][comp] = float(p)
        z = float(np.mean(logit(list(out['components'].values()))))
        cal = self.m['calibrators'][mode]; prob = float(cal.predict_proba(np.array([[z]]))[0, 1])
        kind = threshold_kind or cfg['default_threshold']; thr = cfg['thresholds'][mode][kind]
        out.update(fused_logit=z, probability_male=prob, threshold=thr, threshold_kind=kind, prediction='M' if prob >= thr else 'F', mode=mode,
                   sessions_used=[s['step'] for s in kept])
        return out

    def predict(self, before_video=None, after_video=None, before_meta=None, after_meta=None, use_gate=True, threshold_kind=None):
        ss = []
        if before_video is not None: ss.append(self.session_from_video(before_video, before_meta, 'before'))
        if after_video is not None: ss.append(self.session_from_video(after_video, after_meta, 'after'))
        if not ss: raise ValueError('give at least one video')
        return self.score_sessions(ss, use_gate, threshold_kind)
'''
import importlib.util, sys
from pathlib import Path
Path(DEPLOY).mkdir(parents=True, exist_ok=True); LIB_PATH = Path(DEPLOY)/'sexpred_lib.py'; LIB_PATH.write_text(LIB_SRC)
spec = importlib.util.spec_from_file_location('sexpred_lib', LIB_PATH); L = importlib.util.module_from_spec(spec); sys.modules['sexpred_lib'] = L; spec.loader.exec_module(L)
print('sexpred_lib', L.LIB_VERSION, '->', LIB_PATH)

sexpred_lib 1.0 -> /content/drive/MyDrive/MCD_rPPG/sex_deploy/sexpred_lib.py


## 1. Sessions and paths

In [ ]:
import pickle, shutil, time, warnings, numpy as np, pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **k: x
warnings.filterwarnings('ignore')
DATA_ROOT, TRACE_DIR, INPUTS = Path(ROOT)/'raw', Path(ROOT)/'roi_traces', Path(DEPLOY)/'inputs'
INPUTS.mkdir(parents=True, exist_ok=True)
db_path = next(p for p in (Path(ROOT)/'metadata'/'db.csv', Path(ROOT)/'db.csv') if p.exists())
db = pd.read_csv(db_path)
front = db[db.camera == 'FullHDwebcam'].drop_duplicates(['patient_id', 'step']).sort_values(['patient_id', 'step']).reset_index(drop=True)
if QUICK:
    keep = front.patient_id.drop_duplicates().sample(60, random_state=0); front = front[front.patient_id.isin(keep)].reset_index(drop=True)
front['key'] = front.patient_id.astype(str) + '_' + front.step
trace_path = lambda r: TRACE_DIR / f"{Path(r['video']).name}.npz"
front['has_trace'] = [trace_path(r).exists() for r in front.to_dict('records')]
print(f'{len(front)} front-camera sessions, {front.patient_id.nunique()} people; traces present: {front.has_trace.sum()}')

1200 front-camera sessions, 600 people; traces present: 1200


## 2. Extract missing face traces
Uses the Week 3 extractor: Haar face detection every 10 frames, box smoothed (EMA 0.3), 4 fixed regions, YCrCb skin mask, duplicate-frame flags.

In [ ]:
def get_video(r, tmp):
    src = DATA_ROOT / r['video']
    if src.exists():
        dst = tmp / src.name; shutil.copy(src, dst); return dst
    from huggingface_hub import hf_hub_download   # log in first if downloads are rate-limited: from huggingface_hub import login; login()
    return Path(hf_hub_download('wengziheng/mcd_rppg', r['video'], repo_type='dataset', local_dir=str(tmp)))
def extract_one(r):
    tmp = Path('/content/mcd_tmp' if Path('/content').exists() else './mcd_tmp') / str(os.getpid()); tmp.mkdir(parents=True, exist_ok=True)
    try:
        v = get_video(r, tmp); tr = L.extract_roi_traces(v, tmp / 'out.npz')
        if len(tr['traces']) < 300: raise RuntimeError('fewer than 300 frames')
        shutil.move(str(tmp / 'out.npz'), str(trace_path(r))); return r['key'], 'ok'
    except Exception as e:
        return r['key'], repr(e)[:200]
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
todo = front[~front.has_trace].to_dict('records')
if todo and EXTRACT_MISSING:
    with ProcessPoolExecutor(WORKERS or os.cpu_count(), mp_context=mp.get_context('fork')) as ex:
        res = [f.result() for f in tqdm(as_completed([ex.submit(extract_one, r) for r in todo]), total=len(todo), desc='extract')]
    print(pd.Series([s for _, s in res]).value_counts().to_dict())
    front['has_trace'] = [trace_path(r).exists() for r in front.to_dict('records')]
print(f'Sessions with traces: {front.has_trace.sum()} of {len(front)}')

Sessions with traces: 1200 of 1200


## 3. Build inputs for each recording length
For every session and every length, the trace is cut to the first N seconds *before* any processing, as a deployed short recording would be. Progress is saved every 200 sessions; a rerun continues where it stopped.

In [ ]:
LAB = lambda s: 'full' if s is None else f'{int(s)}s'
def build_one(r):
    try:
        tr = L.load_traces(trace_path(r)); ts = L.parse_meta_timestamps(DATA_ROOT / r['meta']); pw = DATA_ROOT / r['ppg']; out = {}
        for sec in LENGTHS:
            f, raw = L.session_record(tr, ts, sec)
            f['ppg_hr'] = L.contact_hr(pw, max_seconds=sec) if pw.exists() else np.nan
            out[LAB(sec)] = (f, raw)
        return r['key'], out
    except Exception as e:
        return r['key'], repr(e)[:200]
final = {LAB(s): (INPUTS / f'features_{LAB(s)}.csv', INPUTS / f'signals_{LAB(s)}.npz') for s in LENGTHS}
if all(a.exists() and b.exists() for a, b in final.values()):
    print('Inputs already built:', ', '.join(final))
else:
    parts = sorted(INPUTS.glob('part_*.pkl')); done = {}
    for p in parts: done.update(pickle.load(open(p, 'rb')))
    todo = [r for r in front[front.has_trace].to_dict('records') if r['key'] not in done]
    print(f'{len(done)} sessions already built, {len(todo)} to build')
    new, n_part, errors = {}, len(parts), []
    with ProcessPoolExecutor(WORKERS or os.cpu_count(), mp_context=mp.get_context('fork')) as ex:
        for fut in tqdm(as_completed([ex.submit(build_one, r) for r in todo]), total=len(todo), desc='inputs'):
            k, out = fut.result()
            if isinstance(out, str): errors.append((k, out)); continue
            new[k] = out; done[k] = out
            if len(new) >= 200: pickle.dump(new, open(INPUTS / f'part_{n_part:03d}.pkl', 'wb')); n_part += 1; new = {}
    if new: pickle.dump(new, open(INPUTS / f'part_{n_part:03d}.pkl', 'wb'))
    if errors: print('errors:', len(errors), errors[:3])
    for lab, (fcsv, fnpz) in final.items():
        rows, sig = [], {}
        for k in front.key:
            if k not in done: continue
            f, raw = done[k][lab]; rows.append(dict(key=k, **f))
            if raw is not None: sig[k] = raw
        F = pd.DataFrame(rows); F.insert(1, 'patient_id', F.key.str.split('_').str[0].astype(int)); F.insert(2, 'step', F.key.str.split('_').str[1])
        F.to_csv(fcsv, index=False); np.savez_compressed(fnpz, **sig); print(f'{lab}: {len(F)} sessions, {len(sig)} with signals -> {fnpz.name}')
    for p in INPUTS.glob('part_*.pkl'): p.unlink()

0 sessions already built, 1200 to build


inputs:   0%|          | 0/1200 [00:00<?, ?it/s]

errors: 1 [('1386_after', "BadZipFile('File is not a zip file')")]
full: 1199 sessions, 1199 with signals -> signals_full.npz
120s: 1199 sessions, 1199 with signals -> signals_120s.npz
60s: 1199 sessions, 1199 with signals -> signals_60s.npz
30s: 1199 sessions, 1199 with signals -> signals_30s.npz


## 4. Check: does the code reproduce the training data?
- **rPPG features** recomputed here against Week 3's `session_features.csv`, which every model so far used.
- **12-channel signals** against Week 9's `rppg_multi_raw_30hz.npz`.

Both should match almost exactly. A mismatch means the shared code differs from what was trained on.

In [ ]:
FULL = pd.read_csv(INPUTS / 'features_full.csv')
w3 = Path(ROOT)/'week3_stress_rppg600'/'features'/'session_features.csv'
if w3.exists():
    W3 = pd.read_csv(w3); W3['key'] = W3.patient_id.astype(str) + '_' + W3.step
    m = FULL.merge(W3, on='key', suffixes=('', '_w3')); rows = []
    for c in [c for c in FULL.columns if c.startswith('rppg_') and c + '_w3' in m.columns]:
        a, b = m[c].astype(float), m[c + '_w3'].astype(float); both = a.notna() & b.notna()
        rows.append(dict(feature=c, sessions=int(both.sum()), share_equal=float(np.isclose(a[both], b[both], rtol=1e-5, atol=1e-8).mean()) if both.any() else np.nan,
                         max_abs_diff=float((a - b).abs().max()), missing_only_here=int((a.isna() & b.notna()).sum()), missing_only_week3=int((a.notna() & b.isna()).sum())))
    V1 = pd.DataFrame(rows); V1.to_csv(INPUTS / 'verification_features_vs_week3.csv', index=False)
    print(f"rPPG features: {len(V1)} columns compared on {len(m)} sessions; median share equal {V1.share_equal.median():.4f}, lowest {V1.share_equal.min():.4f}")
    display(V1.sort_values('share_equal').head(8).round(5))
else:
    print('Week 3 session_features.csv not found: check skipped.')
w9 = Path(ROOT)/'sex_v3_outputs'/'rppg_multi_raw_30hz.npz'
if w9.exists():
    with np.load(w9) as z9, np.load(INPUTS / 'signals_full.npz') as zn:
        ks = [k for k in zn.files if k in z9.files][:200]
        diffs = [float(np.abs(zn[k].astype(np.float32) - z9[k].astype(np.float32)).max()) if zn[k].shape == z9[k].shape else np.inf for k in ks]
    print(f'12-channel signals vs Week 9 cache ({len(ks)} sessions): identical {np.mean(np.array(diffs) < 1e-2):.1%}, largest difference {max(diffs):.4f}')
else:
    print('Week 9 signal cache not found: check skipped.')

rPPG features: 32 columns compared on 1199 sessions; median share equal 1.0000, lowest 1.0000


,feature,sessions,share_equal,max_abs_diff,missing_only_here,missing_only_week3
0,rppg_face_missing_frac,1199,1.0,0.0,0,0
1,rppg_dup_frac,1199,1.0,0.0,0,0
2,rppg_used_ts,1199,1.0,0.0,0,0
3,rppg_fps_measured,1199,1.0,0.0,0,0
4,rppg_duration_s,1199,1.0,0.0,0,0
5,rppg_snr_face,1199,1.0,0.0,0,0
6,rppg_skin_r,1199,1.0,0.0,0,0
7,rppg_skin_g,1199,1.0,0.0,0,0


12-channel signals vs Week 9 cache (200 sessions): identical 100.0%, largest difference 0.0000


## 5. Summary

In [ ]:
for s in LENGTHS:
    F = pd.read_csv(INPUTS / f'features_{LAB(s)}.csv')
    err = (F.rppg_hr - F.ppg_hr).abs()
    print(f"{LAB(s):>4}: {len(F)} sessions; rPPG HR found {F.rppg_hr.notna().mean():.1%}; HR error vs contact: median {err.median():.2f} BPM, over 5 BPM {(err > 5).mean():.1%}")
print(f'\nShared code: {LIB_PATH}\nInputs: {INPUTS}\nNext: run Week11_Package_rPPG_only.ipynb, then Week11_Package_rPPG_plus_appearance.ipynb.')

full: 1199 sessions; rPPG HR found 100.0%; HR error vs contact: median 0.73 BPM, over 5 BPM 16.1%
120s: 1199 sessions; rPPG HR found 100.0%; HR error vs contact: median 0.66 BPM, over 5 BPM 17.6%
 60s: 1199 sessions; rPPG HR found 100.0%; HR error vs contact: median 0.66 BPM, over 5 BPM 15.7%
 30s: 1199 sessions; rPPG HR found 100.0%; HR error vs contact: median 0.81 BPM, over 5 BPM 23.4%

Shared code: /content/drive/MyDrive/MCD_rPPG/sex_deploy/sexpred_lib.py
Inputs: /content/drive/MyDrive/MCD_rPPG/sex_deploy/inputs
Next: run Week11_Package_rPPG_only.ipynb, then Week11_Package_rPPG_plus_appearance.ipynb.
